# 10 – Hibadiagnosztika és szenzoradat-alapú állapotbecslés

**11. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- process/sensor/actuator fault elkülönítése
- residual képzése referenciajelből
- threshold és EWMA detektor felépítése
- sensitivity/specificity és detection delay értelmezése
- diagnosztikai evidencia Bayes-frissítéshez előkészítése

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Detection – isolation – identification

A diagnosztika három egymásra épülő kérdése: **van-e fault? hol van? milyen típusú/mértékű?**

Példák: process fault (betáplálási rendellenesség), sensor fault (LT-101 bias/drift/stuck), actuator fault (XV-101 nem zár), illetve model fault (a referencia-modell nem írja le jól a valós folyamatot).


## 2. TK-101 szintetikus adatsor

A repository `sensor_data.csv` fájlja oktatási célú szintetikus adat. A `true_level_pct` referencia csak demonstráció; valós rendszerben ‘true’ állapot tipikusan nem ismert közvetlenül.


In [ ]:
path=ROOT/'data'/'pundit_case'/'sensor_data.csv'; df=pd.read_csv(path); df.head()


In [ ]:
plt.plot(df.time_min,df.true_level_pct,label='reference'); plt.plot(df.time_min,df.lt101_measured_pct,label='LT-101',alpha=.8); plt.xlabel('time [min]'); plt.ylabel('level [%]'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 3. Residual

Egyszerű residual: $r(t)=y_{measured}(t)-y_{expected}(t)$. Negatív, növekvő abszolút residual failed-low/bias jellegű hibára utalhat, de residual önmagában nem bizonyítja az okot.


In [ ]:
df['residual']=df.lt101_measured_pct-df.true_level_pct
plt.plot(df.time_min,df.residual); plt.axhline(0,ls='--'); plt.xlabel('time [min]'); plt.ylabel('residual'); plt.grid(alpha=.3); plt.show()


## 4. Threshold detector

Első detektor: fault evidence, ha $r(t)\le -3$. A küszöb szintetikus oktatási paraméter.


In [ ]:
threshold=-3.; df['detected']=df.residual<=threshold
pd.crosstab(df.fault_active.astype(bool),df.detected,rownames=['fault'],colnames=['detected'])


## 5. Diagnosztikai metrikák

$TPR=TP/(TP+FN)$ (sensitivity/recall), $TNR=TN/(TN+FP)$ (specificity). Safety esetben a false negative és false positive költsége nem azonos.


In [ ]:
def metrics(y,p):
 y=np.asarray(y,bool); p=np.asarray(p,bool); tp=np.sum(y&p); tn=np.sum(~y&~p); fp=np.sum(~y&p); fn=np.sum(y&~p)
 return {'TP':tp,'TN':tn,'FP':fp,'FN':fn,'sensitivity':tp/(tp+fn) if tp+fn else np.nan,'specificity':tn/(tn+fp) if tn+fp else np.nan}
metrics(df.fault_active,df.detected)


## 6. Threshold trade-off

Sweepeljük a thresholdot: szigorúbb küszöb csökkentheti a false positive-okat, de növelheti a missed detectiont.


In [ ]:
rows=[]
for th in np.linspace(-.5,-8,50): rows.append({'threshold':th,**metrics(df.fault_active,df.residual<=th)})
perf=pd.DataFrame(rows); plt.plot(perf.threshold,perf.sensitivity,label='sensitivity'); plt.plot(perf.threshold,perf.specificity,label='specificity'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 7. EWMA

$z_t=\alpha r_t+(1-\alpha)z_{t-1}$. Kisebb $\alpha$ simább jelet ad, de késleltetheti a detektálást.


In [ ]:
def ewma(x,alpha=.12):
 x=np.asarray(x,float); z=np.empty_like(x); z[0]=x[0]
 for i in range(1,len(x)): z[i]=alpha*x[i]+(1-alpha)*z[i-1]
 return z
df['ewma']=ewma(df.residual); plt.plot(df.time_min,df.residual,alpha=.35); plt.plot(df.time_min,df.ewma); plt.axhline(-3,ls='--'); plt.grid(alpha=.3); plt.show()


## 8. Detection delay

A sensitivity mellett számít, hogy a fault kezdete után mikor születik evidencia.


In [ ]:
fault_start=df.loc[df.fault_active.eq(1),'time_min'].min(); first=df.loc[df.ewma.le(-3),'time_min'].min(); print('fault start',fault_start,'first detection',first,'delay',first-fault_start)


## 9. Diagnosztikai evidencia → Bayes likelihood

A detector sensitivity és false-positive rate lehet $P(E|F)$ és $P(E|\neg F)$ becslés. Ugyanazon adaton kalibrált és értékelt detector optimista; kutatásban train/validation/test vagy független validáció szükséges.


In [ ]:
m=metrics(df.fault_active,df.detected); sens=m['sensitivity']; fp=1-m['specificity']; prior=.02
post=sens*prior/(sens*prior+fp*(1-prior)) if fp>0 else 1.
print({'sensitivity':sens,'false_positive':fp,'posterior_if_alarm':post})


In [ ]:
from safetycourse.diagnosis import residual,threshold_detector,ewma as lib_ewma,binary_metrics
r=residual(df.lt101_measured_pct,df.true_level_pct); pred=threshold_detector(r,-3.,direction='low'); print(binary_metrics(df.fault_active,pred)); assert np.allclose(lib_ewma(r,.12),df.ewma)


## Próbáld ki!

1. Állíts alpha=0.05, 0.2 és 0.5 értéket és hasonlíts detection delay-t.
2. Válassz thresholdot legalább 0.90 sensitivity mellett, majd add meg specificity-t.
3. Növeld a fault priort 0.10-re és számíts posteriort.
4. Írd le, valós folyamatban mivel helyettesíthető a true_level referencia.


## Összefoglalás

A diagnosztika szenzoradatból evidenciát állít elő. A következő tanegység ezt az evidenciát **időben változó kockázattá és maintenance decision inputtá** alakítja.
